In [ ]:
# Safe setup block
import warnings
warnings.filterwarnings('ignore')

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'


In [ ]:
!pip install -q tensorflow keras-tuner torch torchvision torchaudio nlpaug augly wandb imbalanced-learn keras_cv seaborn matplotlib numpy pandas scikit-learn

In [ ]:
import sys
import tensorflow as tf
import torch

print(f"Python version: {sys.version}")
print(f"TensorFlow version: {tf.__version__}")
print(f"PyTorch version: {torch.__version__}")

print("\n--- GPU Availability ---")
print(f"TF GPU: {len(tf.config.list_physical_devices('GPU')) > 0}")
print(f"PyTorch GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"PyTorch Device Name: {torch.cuda.get_device_name(0)}")


In [ ]:
# --- INLINED UTILS FUNCTION ---
import os
import random
import numpy as np
import tensorflow as tf
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for academic-grade visualizations
sns.set_theme(style="whitegrid", context="talk", palette="deep")

def set_seed(seed=42):
    """
    Ensures absolute reproducibility across all stochastic operations.
    Sets seeds for Python random, NumPy, TensorFlow, and PyTorch.
    """
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # TensorFlow
    tf.random.set_seed(seed)
    
    # PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    print(f"[Utils] Random seed globally set to {seed} for reproducibility.")

def plot_tf_history(history, title="Training History"):
    """
    Plots the accuracy and loss curves for a Keras/TensorFlow training history.
    """
    acc = history.history.get('accuracy', history.history.get('acc', []))
    val_acc = history.history.get('val_accuracy', history.history.get('val_acc', []))
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(1, len(acc) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(title, fontsize=20, fontweight='bold', y=1.05)

    # Accuracy Plot
    ax1.plot(epochs, acc, 'bo-', label='Training Acc', alpha=0.8, linewidth=2)
    ax1.plot(epochs, val_acc, 'ro-', label='Validation Acc', alpha=0.8, linewidth=2)
    ax1.set_title('Accuracy')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Accuracy')
    ax1.legend()

    # Loss Plot
    ax2.plot(epochs, loss, 'bo-', label='Training Loss', alpha=0.8, linewidth=2)
    ax2.plot(epochs, val_loss, 'ro-', label='Validation Loss', alpha=0.8, linewidth=2)
    ax2.set_title('Loss')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Loss')
    ax2.legend()

    plt.tight_layout()
    plt.show()

def plot_torch_history(train_losses, val_losses, train_acc=None, val_acc=None, title="PyTorch Training History"):
    """
    Plots the loss (and optionally accuracy) curves for PyTorch manual training loops.
    """
    epochs = range(1, len(train_losses) + 1)
    
    if train_acc is not None and val_acc is not None:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
        fig.suptitle(title, fontsize=20, fontweight='bold', y=1.05)

        # Accuracy Plot
        ax1.plot(epochs, train_acc, 'bo-', label='Training Acc', alpha=0.8, linewidth=2)
        ax1.plot(epochs, val_acc, 'ro-', label='Validation Acc', alpha=0.8, linewidth=2)
        ax1.set_title('Accuracy')
        ax1.set_xlabel('Epochs')
        ax1.set_ylabel('Accuracy')
        ax1.legend()

        # Loss Plot
        ax2.plot(epochs, train_losses, 'bo-', label='Training Loss', alpha=0.8, linewidth=2)
        ax2.plot(epochs, val_losses, 'ro-', label='Validation Loss', alpha=0.8, linewidth=2)
        ax2.set_title('Loss')
        ax2.set_xlabel('Epochs')
        ax2.set_ylabel('Loss')
        ax2.legend()
    else:
        plt.figure(figsize=(8, 6))
        plt.plot(epochs, train_losses, 'bo-', label='Training Loss', alpha=0.8, linewidth=2)
        plt.plot(epochs, val_losses, 'ro-', label='Validation Loss', alpha=0.8, linewidth=2)
        plt.title(f"{title} - Loss", fontsize=16, fontweight='bold')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend()
        
    plt.tight_layout()
    plt.show()

def compare_models_tf(histories_dict, metric='val_accuracy', title="Model Comparison"):
    """
    Overlays multiple Keras histories on a single plot for A/B testing comparison.
    histories_dict: dict of { 'Label': history_object }
    """
    plt.figure(figsize=(10, 7))
    plt.title(title, fontsize=18, fontweight='bold')
    
    for label, history in histories_dict.items():
        data = history.history.get(metric, history.history.get('val_acc', []))
        epochs = range(1, len(data) + 1)
        plt.plot(epochs, data, marker='o', label=label, linewidth=2, alpha=0.8)
        
    plt.xlabel('Epochs')
    plt.ylabel(metric.replace('_', ' ').title())
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()


# 05. Custom Activation, Initializer, and Regularizer

## 1. Theoretical Background
- **Swish Activation**: $f(x) = x \cdot \sigma(x)$. Found via neural architecture search, often outperforms ReLU because it is smooth and non-monotonic.
- **Custom Orthogonal Initializer**: Forces weight matrices to be orthogonal, preserving the norm of vectors during forward passes.
- **Custom L1-L2 (Elastic Net)**: Combines sparsity (L1) and diffuse weights (L2).

In [ ]:
import tensorflow as tf

# 1. Custom Activation
@tf.keras.utils.register_keras_serializable()
def custom_swish(x):
    return x * tf.math.sigmoid(x)

# 2. Custom Initializer
class CustomOrthogonal(tf.keras.initializers.Initializer):
    def __call__(self, shape, dtype=None, **kwargs):
        flat_shape = (shape[0], tf.math.reduce_prod(shape[1:]))
        a = tf.random.normal(flat_shape, dtype=dtype)
        q, r = tf.linalg.qr(a)
        q = tf.reshape(q, shape)
        return q

# 3. Custom Regularizer
class ElasticNet(tf.keras.regularizers.Regularizer):
    def __init__(self, l1=0.01, l2=0.01):
        self.l1 = l1
        self.l2 = l2
    def __call__(self, x):
        return self.l1 * tf.reduce_sum(tf.abs(x)) + self.l2 * tf.reduce_sum(tf.square(x))